# Lesson 2: LangGraph with Pydantic State

In **Lesson 1** you used `TypedDict` for graph state.  
In this lesson you replace it with **Pydantic** for stronger validation and clearer schemas.

### Scenario: AI Trip Planner

Imagine a small travel assistant that helps a user plan a short trip:

1. The user submits a name, destination preference, budget, and trip length
2. The graph validates the input with Pydantic rules
3. An LLM picks a concrete destination
4. A second LLM drafts a short activity plan
5. A final node formats the itinerary for display

### Graph flow

```
START → validate_input → pick_destination → plan_activities → compose_summary → END
```

### Why Pydantic here?

| Feature | Benefit in LangGraph |
|---------|----------------------|
| `Field(...)` | Defaults, bounds, and documentation on each state key |
| `@field_validator` | Reject bad input before any LLM call |
| `BaseModel` | LangGraph accepts it directly as the state schema |
| Type hints | Nodes receive a real model instance (`state.traveler_name`) |

## 1. Environment configuration

Same `.env` setup as Lesson 1.  
Because this notebook lives in `Lessons/`, we resolve the env file one directory up.

In [ ]:
from pathlib import Path

from dotenv import dotenv_values, load_dotenv

# Lessons/02_...ipynb → project root .env
ENV_PATH = (Path.cwd() / ".." / ".env").resolve()
if not ENV_PATH.exists():
    ENV_PATH = (Path.cwd() / ".env").resolve()

env_file = dotenv_values(ENV_PATH)

keys = ["PROVIDER_URL", "MODEL_NAME", "API_KEY"]
missing_or_empty = [
    key for key in keys if not (env_file.get(key) or "").strip()
]

if missing_or_empty:
    raise ValueError(
        f"Missing or empty in {ENV_PATH}: {', '.join(missing_or_empty)}"
    )

for key in keys:
    print(f".env: {key} is set")

load_dotenv(ENV_PATH, override=True)

.env: PROVIDER_URL is set
.env: MODEL_NAME is set
.env: API_KEY is set


True

## 2. Create the OpenAI client

Two nodes will call the LLM. We create the client once and reuse it.

In [ ]:
from openai import OpenAI

client = OpenAI(
    api_key=env_file["API_KEY"],
    base_url=env_file["PROVIDER_URL"],
)

## 3. Define state with Pydantic

`TripPlannerState` is the **single source of truth** for all data flowing through the graph.

Compared to `TypedDict`:

- Invalid values raise `ValidationError` early (e.g. negative budget)
- `Field` documents each property and sets defaults
- Validators can normalize strings (`strip()` whitespace)

LangGraph nodes receive a **Pydantic instance** and return a **partial dict** of fields to update.

In [ ]:
from pydantic import BaseModel, Field, field_validator


class TripPlannerState(BaseModel):
    """Shared state for the trip-planning workflow."""

    # --- User input ---
    traveler_name: str = Field(description="Name of the traveler")
    destination_hint: str = Field(
        description="Free-text preference, e.g. 'warm beach' or 'historic Europe'"
    )
    budget_usd: int = Field(gt=0, le=50_000, description="Total budget in USD")
    trip_days: int = Field(ge=1, le=30, description="Length of the trip in days")

    # --- Produced by nodes ---
    selected_destination: str = ""
    activity_plan: str = ""
    final_itinerary: str = ""
    status: str = "pending"

    @field_validator("traveler_name", "destination_hint")
    @classmethod
    def strip_and_require_text(cls, value: str) -> str:
        cleaned = value.strip()
        if not cleaned:
            raise ValueError("must not be empty")
        return cleaned

## 4. Implement four nodes

| Node | Role |
|------|------|
| `validate_input` | Confirm state is usable; no LLM call |
| `pick_destination` | Ask the LLM for one concrete city/region |
| `plan_activities` | Ask the LLM for a short day-by-day outline |
| `compose_summary` | Pure Python formatting into a final itinerary |

Each node returns only the fields it changes. LangGraph merges them into the running state.

In [ ]:
def validate_input(state: TripPlannerState) -> dict:
    """Node 1: Pydantic already validated types; mark the request as ready."""

    print(
        f"Validated trip request for {state.traveler_name}: "
        f"{state.destination_hint}, ${state.budget_usd}, {state.trip_days} days"
    )
    return {"status": "validated"}


def pick_destination(state: TripPlannerState) -> dict:
    """Node 2: Use the LLM to choose a specific destination."""

    prompt = (
        f"Traveler preference: {state.destination_hint}.\n"
        f"Budget: ${state.budget_usd} USD for {state.trip_days} days.\n"
        "Reply with ONE city or region name only, no explanation."
    )

    response = client.chat.completions.create(
        model=env_file["MODEL_NAME"],
        messages=[{"role": "user", "content": prompt}],
        max_tokens=40,
        reasoning_effort="none",
    )

    destination = response.choices[0].message.content.strip()
    print(f"Selected destination: {destination}")

    return {
        "selected_destination": destination,
        "status": "destination_selected",
    }


def plan_activities(state: TripPlannerState) -> dict:
    """Node 3: Ask the LLM for a concise activity outline."""

    prompt = (
        f"Create a brief {state.trip_days}-day activity outline for "
        f"{state.selected_destination}. Budget about ${state.budget_usd}. "
        "Use 3-5 bullet points, keep it practical."
    )

    response = client.chat.completions.create(
        model=env_file["MODEL_NAME"],
        messages=[{"role": "user", "content": prompt}],
        max_tokens=300,
        reasoning_effort="none",
    )

    activities = response.choices[0].message.content.strip()
    print("Activity plan drafted.")

    return {
        "activity_plan": activities,
        "status": "activities_planned",
    }


def compose_summary(state: TripPlannerState) -> dict:
    """Node 4: Format the final itinerary without calling the LLM."""

    itinerary = (
        f"Trip itinerary for {state.traveler_name}\n"
        f"{'=' * 40}\n"
        f"Preference: {state.destination_hint}\n"
        f"Destination: {state.selected_destination}\n"
        f"Duration: {state.trip_days} days | Budget: ${state.budget_usd}\n\n"
        f"Suggested activities:\n{state.activity_plan}"
    )

    return {
        "final_itinerary": itinerary,
        "status": "complete",
    }

## 5. Build the graph

Pass the Pydantic class directly to `StateGraph` — no wrapper needed.

We wire four nodes in a straight line. Later lessons will add **conditional edges** (e.g. skip LLM if budget is too low).

In [ ]:
from langgraph.graph import END, START, StateGraph

graph = StateGraph(TripPlannerState)

graph.add_node("validate_input", validate_input)
graph.add_node("pick_destination", pick_destination)
graph.add_node("plan_activities", plan_activities)
graph.add_node("compose_summary", compose_summary)

graph.add_edge(START, "validate_input")
graph.add_edge("validate_input", "pick_destination")
graph.add_edge("pick_destination", "plan_activities")
graph.add_edge("plan_activities", "compose_summary")
graph.add_edge("compose_summary", END)

## 6. Compile and visualize

With four nodes the diagram is more interesting than Lesson 1's single-node graph.

In [ ]:
from IPython.display import Image, display

trip_planner = graph.compile()

display(Image(trip_planner.get_graph().draw_mermaid_png()))

# print(trip_planner.get_graph().draw_mermaid())

## 7. Invoke the graph

Pass a plain **dict** matching `TripPlannerState`.  
Pydantic validates it when the graph starts.

The return value is a **dict** with the final merged state.

In [ ]:
result = trip_planner.invoke(
    {
        "traveler_name": "Sajjad",
        "destination_hint": "warm beach with good food",
        "budget_usd": 2000,
        "trip_days": 5,
    }
)

print(result["status"])
print()
print(result["final_itinerary"])

## 8. See Pydantic validation in action

Try invalid input — the graph fails **before** any node runs because Pydantic rejects the state.

In [ ]:
from pydantic import ValidationError

try:
    trip_planner.invoke(
        {
            "traveler_name": "Sajjad",
            "destination_hint": "mountains",
            "budget_usd": -500,  # invalid: must be > 0
            "trip_days": 5,
        }
    )
except ValidationError as exc:
    print("Pydantic blocked invalid state:")
    print(exc.errors()[0]["msg"])

## Recap

- **Pydantic `BaseModel`** works as a LangGraph state schema
- Nodes get a **model instance**; return a **dict** of changed fields
- **`invoke({...})`** accepts a dict; output is also a dict
- Validators and `Field` constraints catch bad input early
- Mix **LLM nodes** and **pure Python nodes** in the same graph

**Next:** conditional routing, reducers for list fields, and checkpointing.